In [4]:
!pip3 install requests pandas tqdm -q

import requests
import pandas as pd
from tqdm import tqdm
import time



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
!pip3 install requests pandas tqdm -q

import requests
import pandas as pd
from tqdm import tqdm
import time

API_KEY = "f2d68b86-4c0b-4fe7-87ef-c792cd510f29"  # paste your key here
BASE_URL = "https://api.company-information.service.gov.uk"

# Test the key works
test = requests.get(
    f"{BASE_URL}/advanced-search/companies",
    params={"sic_codes": "62020", "company_status": "active", "size": 1},
    auth=(API_KEY, "")
)
print(f"Status: {test.status_code}")
print(f"Sample: {test.json()}")

In [ ]:
SIC_CODES = [
    "62020",
    "62012",
    "62090",
    "63120",
    "62011",
    "63990",
    "66190",
    "64999",
]

def fetch_companies_by_sic(sic_code, max_results=10000):
    companies = []
    start_index = 0
    page_size = 100

    with tqdm(total=max_results, desc=f"SIC {sic_code}") as pbar:
        while len(companies) < max_results:
            response = requests.get(
                f"{BASE_URL}/advanced-search/companies",
                params={
                    "sic_codes": sic_code,
                    "company_status": "active",
                    "size": page_size,
                    "start_index": start_index
                },
                auth=(API_KEY, "")
            )

            if response.status_code == 429:
                print("Rate limited — waiting 10 seconds...")
                time.sleep(10)
                continue

            if response.status_code != 200:
                print(f"Error {response.status_code} for SIC {sic_code}")
                break

            data = response.json()
            items = data.get("items", [])
            if not items:
                print(f"  No more results at index {start_index}")
                break

            companies.extend(items)
            pbar.update(len(items))
            start_index += page_size
            time.sleep(0.5)

            if len(items) < page_size:
                print(f"  Reached end at {len(companies)} companies")
                break

    return companies[:max_results]

all_companies = []
for sic in SIC_CODES:
    results = fetch_companies_by_sic(sic, max_results=10000)
    print(f"SIC {sic}: {len(results)} companies fetched")
    all_companies.extend(results)

print(f"\nTotal before dedup: {len(all_companies)}")

SIC codes:  12%|█▎        | 1/8 [04:35<32:08, 275.49s/it]

SIC 62020: 10000 companies


SIC codes:  25%|██▌       | 2/8 [04:39<11:36, 116.06s/it]

SIC 63120: 10000 companies


SIC codes:  38%|███▊      | 3/8 [04:41<05:19, 63.98s/it] 

...

SIC 63990: 10000 companies


SIC codes:  88%|████████▊ | 7/8 [09:21<00:56, 56.40s/it]

SIC 66190: 10000 companies


SIC codes: 100%|██████████| 8/8 [09:22<00:00, 70.33s/it]

SIC 64999: 10000 companies

Total before dedup: 80000


In [ ]:
def parse_company(c):
    address = c.get("registered_office_address", {})
    sic_codes = c.get("sic_codes", [])
    return {
        "CompanyName": c.get("company_name", ""),
        "CompanyNumber": c.get("company_number", ""),
        "CompanyStatus": c.get("company_status", ""),
        "RegAddress.PostTown": address.get("locality", ""),
        "RegAddress.PostCode": address.get("postal_code", ""),
        "RegAddress.Country": address.get("country", ""),
        "SICCode.SicText_1": sic_codes[0] if sic_codes else "",
        "IncorporationDate": c.get("date_of_creation", ""),
    }

df = pd.DataFrame([parse_company(c) for c in all_companies])

# Drop duplicates (same company might appear in multiple SIC queries)
df = df.drop_duplicates(subset="CompanyNumber").reset_index(drop=True)
print(f"Unique companies: {len(df)}")
print(df.head())

save_path = 'companies_api.csv'
df.to_csv(save_path, index=False)
print("Saved companies_api.csv")

Unique companies: 74447
                             CompanyName CompanyNumber CompanyStatus  \
0                    H J WHEELER LIMITED      07394977        active   
1  RICHWIND SOFTWARE DEVELOPMENT LIMITED      07438022        active   
2              KENILWORTH CONSULTING LTD      07256308        active   
3                   TOLLINGTON DEYES LTD      06799271        active   
4                      MICHAEL W LIMITED      09982337        active   

  RegAddress.PostTown RegAddress.PostCode RegAddress.Country  \
0             Newbury            RG14 7NE                      
1              London             E15 4DJ                      
2            Loughton            IG10 3AG                      
3          Winchester            SO23 7RD                      
4          Manchester              M3 3HF            England   

  SICCode.SicText_1 IncorporationDate  
0             62020        2010-10-04  
1             62020        2010-11-12  
2             62020        2010-05-18 

In [ ]:
import pandas as pd

df = pd.read_csv('companies_api.csv')

print(f"Total rows: {len(df)}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst SIC value: '{df['SICCode.SicText_1'].iloc[0]}' type: {type(df['SICCode.SicText_1'].iloc[0])}")
print(f"\nUnique SIC values (first 10):")
print(df['SICCode.SicText_1'].unique()[:10])
print(f"\nIncorporationDate sample:")
print(df['IncorporationDate'].head())

In [ ]:
import pandas as pd

df = pd.read_csv('companies_api.csv')

# SIC codes are integers - compare as integers
target_sics = [62020, 62012, 62090, 63120, 62011, 63990, 66190, 64999]
df = df[df['SICCode.SicText_1'].isin(target_sics)].reset_index(drop=True)
print(f"After SIC filter: {len(df)}")

# Standardise text
df['CompanyName'] = df['CompanyName'].str.upper()
df['RegAddress.PostTown'] = df['RegAddress.PostTown'].str.upper()

# Filter incorporated after 2010
df['IncorporationDate'] = pd.to_datetime(df['IncorporationDate'], errors='coerce')
df = df[df['IncorporationDate'] >= '2010-01-01'].reset_index(drop=True)
print(f"After date filter: {len(df)}")

print(df['SICCode.SicText_1'].value_counts())

save_path = 'companies_api_clean.csv'
df.to_csv(save_path, index=False)
print("Saved companies_api_clean.csv")

In [6]:
!pip3 install requests beautifulsoup4 tqdm -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import requests
import pandas as pd
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

df = pd.read_csv('companies_api_clean.csv')
print(f"Working with {len(df)} companies")

def clean_company_name(name):
    name = name.lower()
    removals = ['limited','ltd','plc','llp','llc','group','holdings',
                'consulting','consultancy','solutions','services',
                'technologies','technology','digital','software',
                'systems','uk','global']
    for word in removals:
        name = re.sub(r'\b' + word + r'\b', '', name)
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', '', name).strip()
    return name

def guess_domains(company_name):
    clean = clean_company_name(company_name)
    if not clean or len(clean) < 3:
        return []
    return [
        f"https://www.{clean}.co.uk",
        f"https://www.{clean}.com",
        f"https://{clean}.io",
    ]

def find_website(row):
    for url in guess_domains(row['CompanyName']):
        try:
            r = requests.head(url, timeout=4, allow_redirects=True)
            if r.status_code < 400:
                return (row.name, url)
        except:
            continue
    return (row.name, None)

results = {}
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(find_website, row): row.name
               for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df)):
        idx, url = future.result()
        results[idx] = url

df['website'] = df.index.map(results)
found = df['website'].notna().sum()
print(f"\nWebsites found: {found} / {len(df)} ({round(found/len(df)*100)}%)")

df.to_csv('companies_with_websites.csv', index=False)
print("Saved companies_with_websites.csv")

Working with 52231 companies


 24%|██▍       | 12703/52231 [17:53<55:40, 11.83it/s]  


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

df = pd.read_csv('companies_with_websites.csv')
df_with = df[df['website'].notna()].copy().reset_index(drop=True)
print(f"Verifying {len(df_with)} companies...")

def get_page_text(url, timeout=6):
    try:
        r = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'
        })
        soup = BeautifulSoup(r.text, 'html.parser')
        return soup.get_text(separator=' ', strip=True).lower()
    except:
        return ''

def verify_website(row):
    url = row['website']
    company = row['CompanyName'].lower()
    town = str(row['RegAddress.PostTown']).lower()
    postcode = str(row['RegAddress.PostCode']).lower().split()[0]

    clean = re.sub(r'\b(limited|ltd|plc|llp|consulting|solutions|services|technologies|technology|digital|software|systems|group)\b', '', company)
    clean = re.sub(r'[^a-z0-9\s]', '', clean).strip()
    keywords = [w for w in clean.split() if len(w) > 3]

    text = get_page_text(url)
    if not text:
        return (row.name, False)

    domain = url.replace('https://', '').replace('http://', '').replace('www.', '').split('.')[0]
    domain_match = any(kw in domain for kw in keywords)
    keyword_matches = sum(1 for kw in keywords if kw in text)
    keyword_match = keyword_matches >= max(1, len(keywords) // 2)
    location_match = town in text or postcode in text

    verified = domain_match and (keyword_match or location_match)
    return (row.name, verified)

results = {}
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(verify_website, row): row.name
               for _, row in df_with.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df_with)):
        idx, verified = future.result()
        results[idx] = verified

df_with['verified'] = df_with.index.map(results)
verified_df = df_with[df_with['verified'] == True].reset_index(drop=True)

print(f"\nVerified: {len(verified_df)} / {len(df_with)}")
print(f"Rejected: {len(df_with) - len(verified_df)}")

verified_df.to_csv('companies_verified.csv', index=False)
print("Saved companies_verified.csv")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_verified.csv')
print(f"Scraping {len(df)} verified companies...")

def scrape_website(row):
    url = row['website']
    try:
        r = requests.get(url, timeout=8, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'
        })
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['nav','footer','header','script','style','aside']):
            tag.decompose()
        title = soup.title.string.strip() if soup.title else ''
        meta = soup.find('meta', attrs={'name': 'description'})
        meta_desc = meta['content'].strip() if meta and meta.get('content') else ''
        body = soup.get_text(separator=' ', strip=True)
        body = ' '.join(body.split())[:2000]
        full_text = f"{title}. {meta_desc}. {body}"
        return (row.name, full_text)
    except:
        return (row.name, None)

results = {}
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(scrape_website, row): row.name
               for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df)):
        idx, text = future.result()
        results[idx] = text

df['scraped_text'] = df.index.map(results)
before = len(df)
df = df[df['scraped_text'].notna()].reset_index(drop=True)
print(f"\nSuccessfully scraped: {len(df)} / {before}")

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_scraped.csv', index=False)
print("Saved companies_scraped.csv")

In [ ]:
import pandas as pd
import re
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_scraped.csv')

def clean_text(text):
    if not isinstance(text, str):
        return None
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    boilerplate = ['cookie','privacy policy','terms of service',
                   'all rights reserved','copyright','gdpr',
                   'we use cookies','accept cookies','sign in','log in']
    for phrase in boilerplate:
        text = text.replace(phrase, '')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 100 else None

df['clean_text'] = df['scraped_text'].apply(clean_text)
before = len(df)
df = df[df['clean_text'].notna()].reset_index(drop=True)
print(f"Dropped: {before - len(df)}")
print(f"Clean dataset: {len(df)} companies")

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clean.csv', index=False)
print("Saved companies_clean.csv")

In [1]:
!pip3 install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

df = pd.read_csv('companies_clean.csv')
print(f"Embedding {len(df)} companies...")

model = SentenceTransformer('all-MiniLM-L6-v2')
texts = df['clean_text'].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

np.save('embeddings.npy', embeddings)

df_meta = df.drop(columns=['scraped_text', 'clean_text'])
df_meta.to_csv('companies_meta.csv', index=False)

print("Saved embeddings.npy and companies_meta.csv")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Embedding 13177 companies...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/206 [00:00<?, ?it/s]

Embeddings shape: (13177, 384)
Saved embeddings.npy and companies_meta.csv


In [2]:
!pip3 install hdbscan umap-learn plotly -q

import numpy as np
import pandas as pd
import hdbscan
import umap

embeddings = np.load('embeddings.npy')
df = pd.read_csv('companies_meta.csv')

print("UMAP 10D for clustering...")
reducer_10d = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
embedding_10d = reducer_10d.fit_transform(embeddings)

print("Clustering with HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(embedding_10d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print(f"\nClusters: {n_clusters}")
print(f"Noise: {noise}")
print(f"Clustered: {len(df) - noise}")
print(pd.Series(labels).value_counts().sort_index())

print("\nUMAP 2D for visualisation...")
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
embedding_2d = reducer_2d.fit_transform(embeddings)

df['x'] = embedding_2d[:, 0]
df['y'] = embedding_2d[:, 1]
df['cluster'] = labels

df.to_csv('companies_clustered.csv', index=False)
print("Saved companies_clustered.csv")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
UMAP 10D for clustering...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Clustering with HDBSCAN...

Clusters: 94
Noise: 4926
Clustered: 8251
-1     4926
 0       26
 1       35
 2       25
 3      361
       ... 
 89     379
 90      28
 91      46
 92      41
 93     128
Name: count, Length: 95, dtype: int64

UMAP 2D for visualisation...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved companies_clustered.csv


In [3]:
import numpy as np
import pandas as pd
import hdbscan
import umap

embeddings = np.load('embeddings.npy')
df = pd.read_csv('companies_meta.csv')

# Already computed — reload 10D embedding
print("UMAP 10D...")
reducer_10d = umap.UMAP(
    n_components=10, n_neighbors=15,
    min_dist=0.0, metric='cosine', random_state=42
)
embedding_10d = reducer_10d.fit_transform(embeddings)

# Increase min_cluster_size for larger dataset
print("Clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(embedding_10d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print(f"Clusters: {n_clusters}")
print(f"Noise: {noise} ({round(noise/len(df)*100)}%)")
print(pd.Series(labels).value_counts().sort_index())

UMAP 10D...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Clustering...
Clusters: 18
Noise: 2429 (18%)
-1     2429
 0      172
 1      377
 2      361
 3      190
 4      599
 5      171
 6      176
 7      370
 8     5473
 9      513
 10     411
 11     414
 12     166
 13     206
 14     190
 15     546
 16     240
 17     173
Name: count, dtype: int64


In [4]:
import umap
import plotly.express as px
import numpy as np
import pandas as pd

embeddings = np.load('embeddings.npy')
df = pd.read_csv('companies_meta.csv')

print("UMAP 2D for visualisation...")
reducer_2d = umap.UMAP(
    n_components=2, n_neighbors=15,
    min_dist=0.1, metric='cosine', random_state=42
)
embedding_2d = reducer_2d.fit_transform(embeddings)

df['x'] = embedding_2d[:, 0]
df['y'] = embedding_2d[:, 1]
df['cluster'] = labels
df['cluster_label'] = df['cluster'].apply(
    lambda x: f"Cluster {x}" if x != -1 else "Unclustered"
)

df.to_csv('companies_clustered.csv', index=False)

fig = px.scatter(
    df[df['cluster'] != -1],
    x='x', y='y',
    color='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'SICCode.SicText_1': True,
        'website': True,
        'x': False, 'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200, height=800
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(legend_title_text='Cluster')

fig.write_html('clusters_final.html')
print("Saved clusters_final.html")
print(f"\nFinal summary:")
print(f"Total companies: {len(df)}")
print(f"Clusters: 18")
print(f"Clustered: {(df['cluster'] != -1).sum()}")
print(f"Noise: {(df['cluster'] == -1).sum()}")

UMAP 2D for visualisation...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved clusters_final.html

Final summary:
Total companies: 13177
Clusters: 18
Clustered: 10748
Noise: 2429


In [5]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('companies_clustered.csv')

labels = {
    -1: 'Unclustered',
     0: 'Web and digital services',
     1: 'Generic IT consultancy',
     2: 'Cybersecurity and data',
     3: 'Data and strategy consulting',
     4: 'International IT services',
     5: 'Software development studios',
     6: 'IT training and education',
     7: 'Financial services and fintech',
     8: 'General IT and software',
     9: 'Business consulting and trading',
    10: 'Health tech and wellness',
    11: 'UK regional consultancies',
    12: 'Strategy and research',
    13: 'Global business services',
    14: 'Creative and lifestyle tech',
    15: 'Professional services',
    16: 'Financial software and lending',
    17: 'IT outsourcing and managed services',
}

df['cluster_label'] = df['cluster'].map(labels)

fig = px.scatter(
    df[df['cluster'] != -1],
    x='x', y='y',
    color='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'SICCode.SicText_1': True,
        'website': True,
        'x': False, 'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200, height=800
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(legend_title_text='Cluster')

df.to_csv('companies_final.csv', index=False)
fig.write_html('clusters_final_labelled.html')
print("Saved companies_final.csv and clusters_final_labelled.html")

Saved companies_final.csv and clusters_final_labelled.html


In [8]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('companies_final.csv')

# Drop unclustered points to reduce noise visually
df_plot = df[df['cluster'] != -1].copy()

fig = px.scatter(
    df_plot,
    x='x', y='y',
    color='cluster_label',
    hover_name='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'website': True,
        'SICCode.SicText_1': True,
        'x': False,
        'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200,
    height=800
)

fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    legend_title_text='Cluster',
    legend=dict(
        itemsizing='constant',
        font=dict(size=12)
    ),
    hoverlabel=dict(
        bgcolor='white',
        font_size=13
    )
)

fig.write_html('clusters_final_labelled.html')
print("Done")

Done
